# Silver - Cleaning Data

This notebook reads the `bronze` tables, cleans each one step by step, and
loads the result into the `silver` schema.

Run the cells from top to bottom.


In [1]:
import os

import pandas as pd
from sqlalchemy import create_engine, text

DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# Used when checking for future dates.
today = pd.Timestamp.today().normalize()


## Setup

In [2]:
def get_bronze(table_name):
    '''Read one table from the bronze schema.'''
    return pd.read_sql(f"select * from bronze.{table_name}", engine)


### Helper functions

These small functions are reused for every table.


In [3]:
# Text values that should be treated as missing.
INVALID_TEXT = ["", "-", "unknown", "nan", "none", "n/a", "na", "null"]


def clean_text(column):
    '''Trim spaces and turn placeholder text into missing values.'''
    text = column.astype("string").str.strip()
    text.loc[text.str.lower().isin(INVALID_TEXT)] = pd.NA
    return text


def clean_id(column, pattern):
    '''Uppercase an ID and blank out values that do not match the pattern.'''
    ids = column.astype("string").str.strip().str.upper()
    valid = ids.str.match(pattern, na=False)
    ids.loc[~valid] = pd.NA
    return ids


def remove_duplicates(df, id_column):
    '''Keep one row per ID, but keep rows that have no ID.'''
    missing_ids = df[df[id_column].isna()]
    has_ids = df[df[id_column].notna()]
    has_ids = has_ids.drop_duplicates(subset=id_column, keep="first")
    return pd.concat([has_ids, missing_ids], ignore_index=True)


In [4]:
def parse_dates(column):
    '''Read both 2025-10-08 and 08/10/2025 style dates.'''
    text = column.astype("string").str.strip()

    # Read dates like 2025-10-08 first, so day and month are not swapped.
    dates = pd.to_datetime(text, errors="coerce", format="mixed")

    # Then read dates like 08/10/2025 day-first.
    day_first = text.str.match(r"^\d{1,2}[/-]\d{1,2}[/-]\d{4}")
    dates.loc[day_first] = pd.to_datetime(
        text[day_first], errors="coerce", format="mixed", dayfirst=True
    )
    return dates


def to_number(column, unit=None, keep_currency=False):
    '''Turn text like '1,250 kg' or 'INR 2,200' into a number.'''
    text = column.astype("string").str.strip().str.lower()
    text = text.str.replace(",", "", regex=False)

    if unit is not None:
        text = text.str.replace(unit, "", regex=False)

    if keep_currency:
        text = text.str.replace("inr", "", regex=False)
        text = text.str.replace("₹", "", regex=False)
    else:
        # A price in a weight or capacity column is a data-entry mistake.
        wrong_currency = (
            text.str.contains("inr", na=False)
            | text.str.contains("₹", na=False)
        )
        text.loc[wrong_currency] = pd.NA

    return pd.to_numeric(text, errors="coerce")


In [5]:
# Different spellings that mean the same city.
CITY_ALIASES = {
    "Banaras": "Varanasi",
    "Benares": "Varanasi",
    "Bangalore": "Bengaluru",
    "Banglore": "Bengaluru",
    "Bengalore": "Bengaluru",
    "Baroda": "Vadodara",
    "Bombay": "Mumbai",
    "Mumbai City": "Mumbai",
    "Calcutta": "Kolkata",
    "Cochin": "Kochi",
    "Dilli": "Delhi",
    "New Delhi": "Delhi",
    "Gurgaon": "Gurugram",
    "Hubli": "Hubballi",
    "Madras": "Chennai",
    "Trivandram": "Thiruvananthapuram",
    "Trivandrum": "Thiruvananthapuram",
}

# Short state codes that should be expanded.
STATE_ALIASES = {
    "Ap": "Andhra Pradesh",
    "As": "Assam",
    "Br": "Bihar",
    "Ch": "Chandigarh",
    "Dl": "Delhi",
    "Gj": "Gujarat",
    "Hr": "Haryana",
    "Jh": "Jharkhand",
    "Ka": "Karnataka",
    "Kl": "Kerala",
    "Mh": "Maharashtra",
    "Mp": "Madhya Pradesh",
    "Od": "Odisha",
    "Pb": "Punjab",
    "Rj": "Rajasthan",
    "Tg": "Telangana",
    "Tn": "Tamil Nadu",
    "Uk": "Uttarakhand",
    "Up": "Uttar Pradesh",
    "Wb": "West Bengal",
}


def clean_city(column):
    city = clean_text(column).str.title()
    return city.replace(CITY_ALIASES)


def clean_state(column):
    state = clean_text(column).str.title()
    return state.replace(STATE_ALIASES)


def clean_phone(column):
    # Keep digits only, then keep the last 10 digits.
    phone = column.astype("string").str.replace(r"[^0-9]", "", regex=True)
    phone = phone.str[-10:]

    # Indian mobile numbers start with 6, 7, 8 or 9.
    valid = phone.str.match(r"^[6-9]\d{9}$", na=False)
    phone.loc[~valid] = pd.NA
    return phone


def clean_email(column):
    email = column.astype("string").str.strip().str.lower()
    # An email needs text, one @, and a dot after the @.
    valid = email.str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", na=False)
    email.loc[~valid] = pd.NA
    return email


In [6]:
def load_to_silver(df, table_name):
    '''Replace the silver table with the cleaned DataFrame.'''
    df.to_sql(
        table_name,
        engine,
        schema="silver",
        if_exists="replace",
        index=False,
    )
    print("Loaded silver." + table_name)


## Customers

In [7]:
df_customers = get_bronze("customers")

print("Rows:", len(df_customers))
print("Columns:", list(df_customers.columns))
df_customers.head()


Rows: 15420
Columns: ['customer_id', 'customer_name', 'gender', 'customer_type', 'email', 'phone', 'city', 'state', 'pincode', 'signup_date']


,customer_id,customer_name,gender,customer_type,email,phone,city,state,pincode,signup_date
0,CUST004292,Azaan Wason,Male,Individual,azaan.wason793@hotmail.com,6256133336,Coimbatore,Tamil Nadu,497914,2025-01-18
1,CUST007875,Zilmil Sood,F,Individual,zilmil.sood573@gmail.com,6585811228,Vijayawada,Andhra Pradesh,494630,2025-03-26
2,CUST003326,Teerth Bhasin,M,Individual,teerth.bhasin707@yahoo.com,8702177025,Vadodara,Gujarat,322146,2025-04-26
3,CUST004049,Hemani Chandran,F,Individual,hemani.chandran785@outlook.com,6971718231,Udaipur,Rajasthan,312848,2025-11-06
4,CUST000736,Nidhi Dara,F,Individual,nidhi.dara825@outlook.com,7070204262,Ranchi,Jharkhand,809048,2023-01-24


In [8]:
# Customer IDs look like CUST004292.
df_customers["customer_id"] = clean_id(
    df_customers["customer_id"], r"^CUST\d{6}$"
)
df_customers = remove_duplicates(df_customers, "customer_id")

print("Rows after removing duplicates:", len(df_customers))
print("Rows with no customer ID:", df_customers["customer_id"].isna().sum())


Rows after removing duplicates: 15004
Rows with no customer ID: 225


In [9]:
gender = df_customers["gender"].astype("string").str.strip().str.lower()
gender = gender.replace({
    "m": "Male",
    "male": "Male",
    "f": "Female",
    "female": "Female",
    "other": "Others",
    "others": "Others",
})
gender.loc[~gender.isin(["Male", "Female", "Others"])] = pd.NA
df_customers["gender"] = gender

print(df_customers["gender"].value_counts(dropna=False))


gender
Female    7391
Male      7320
<NA>       293
Name: count, dtype: Int64


In [10]:
df_customers["email"] = clean_email(df_customers["email"])

print("Null emails:", df_customers["email"].isna().sum())
print(df_customers["email"].str.split("@").str[-1].value_counts())


Null emails: 439
email
rediffmail.com    2992
yahoo.com         2922
hotmail.com       2897
gmail.com         2897
outlook.com       2857
Name: count, dtype: int64


In [11]:
df_customers["phone"] = clean_phone(df_customers["phone"])

print("Valid phone numbers:", df_customers["phone"].notna().sum())
print("Null phone numbers:", df_customers["phone"].isna().sum())


Valid phone numbers: 14726
Null phone numbers: 278


In [12]:
df_customers["city"] = clean_city(df_customers["city"])

print(df_customers["city"].value_counts(dropna=False))


city
Delhi                 652
Udaipur               369
Amritsar              367
Thiruvananthapuram    366
Faridabad             356
Varanasi              354
Chandigarh            351
Lucknow               351
Jodhpur               349
Guwahati              346
Kolkata               346
Kochi                 344
Mysuru                344
Indore                343
Coimbatore            342
Jaipur                340
Vadodara              339
Agra                  339
Ahmedabad             338
Kanpur                338
Bhubaneswar           337
Ludhiana              335
Visakhapatnam         333
Pune                  331
Vijayawada            330
Hubballi              330
Bhopal                330
Hyderabad             330
Surat                 329
Gurugram              327
Chennai               327
Siliguri              325
Mumbai                324
Ranchi                320
Rajkot                317
Nashik                317
Nagpur                317
Bengaluru             316
Howrah 

In [13]:
df_customers["state"] = clean_state(df_customers["state"])

print(df_customers["state"].value_counts(dropna=False))


state
Uttar Pradesh     1650
Gujarat           1300
Maharashtra       1268
Rajasthan         1046
Karnataka          995
Tamil Nadu         979
West Bengal        973
Kerala             711
Punjab             706
Haryana            690
Andhra Pradesh     669
Madhya Pradesh     660
Delhi              654
Telangana          642
Chandigarh         352
Assam              350
Odisha             342
Jharkhand          323
Bihar              304
Uttarakhand        298
<NA>                92
Name: count, dtype: Int64


In [14]:
# Keep only digits, then check for a valid 6-digit pincode.
pincode = df_customers["pincode"].astype("string").str.replace(
    r"[^0-9]", "", regex=True
)
valid = pincode.str.match(r"^[1-9]\d{5}$", na=False)
pincode.loc[~valid] = pd.NA
df_customers["pincode"] = pincode

print("Null pincodes:", df_customers["pincode"].isna().sum())


Null pincodes: 302


In [15]:
df_customers["signup_date"] = parse_dates(df_customers["signup_date"])

print("Null signup dates:", df_customers["signup_date"].isna().sum())
print(
    "Date range:",
    df_customers["signup_date"].min(),
    "to",
    df_customers["signup_date"].max(),
)


Null signup dates: 160
Date range: 2023-01-01 00:00:00 to 2027-12-20 00:00:00


## Delivery events

In [16]:
df_delivery_events = get_bronze("delivery_events")

print("Rows:", len(df_delivery_events))
print("Columns:", list(df_delivery_events.columns))
df_delivery_events.head()


Rows: 42755
Columns: ['event_id', 'shipment_id', 'event_type', 'event_timestamp', 'location_city', 'remarks']


,event_id,shipment_id,event_type,event_timestamp,location_city,remarks
0,EVT0030840,SHP0029298,Delivery Failed,2023-11-03 22:54:35,New Delhi,On time
1,EVT0009572,SHP0062388,In Transit,2024-02-21 22:29:54,Surat,Rescheduled
2,EVT0037322,SHP0059122,Delivery Failed,2023-12-02 17:41:06,Agra,Customer unavailable
3,EVT0030645,SHP0085010,Picked Up,2024-09-25 18:18:20,Patna,Rescheduled
4,EVT0006113,SHP0036922,Returned to Warehouse,2024-06-24 16:39:33,Dehradun,Traffic delay


In [17]:
# Event IDs look like EVT0030840.
df_delivery_events["event_id"] = clean_id(
    df_delivery_events["event_id"], r"^EVT\d{7}$"
)
df_delivery_events = remove_duplicates(df_delivery_events, "event_id")

print("Rows after removing duplicates:", len(df_delivery_events))


Rows after removing duplicates: 42000


In [18]:
# Shipment IDs look like SHP0059521.
df_delivery_events["shipment_id"] = clean_id(
    df_delivery_events["shipment_id"], r"^SHP\d{7}$"
)

print("Null shipment IDs:", df_delivery_events["shipment_id"].isna().sum())


Null shipment IDs: 540


In [19]:
event_type = df_delivery_events["event_type"].astype("string").str.strip().str.lower()
event_type = event_type.str.replace("-", " ", regex=False)
event_type = event_type.replace({
    "delivery failed": "Delivery Failed",
    "failed": "Delivery Failed",
    "in transit": "In Transit",
    "picked up": "Picked Up",
    "pickedup": "Picked Up",
    "returned": "Returned to Warehouse",
    "returned to warehouse": "Returned to Warehouse",
    "delivered": "Delivered",
    "deliverd": "Delivered",
    "out for delivery": "Out for Delivery",
})

valid_event_types = [
    "Delivery Failed",
    "In Transit",
    "Picked Up",
    "Returned to Warehouse",
    "Delivered",
    "Out for Delivery",
]
event_type.loc[~event_type.isin(valid_event_types)] = pd.NA
df_delivery_events["event_type"] = event_type

print(df_delivery_events["event_type"].value_counts(dropna=False))


event_type
Delivery Failed          7071
Out for Delivery         6973
Picked Up                6970
Delivered                6960
In Transit               6927
Returned to Warehouse    6910
<NA>                      189
Name: count, dtype: Int64


In [20]:
df_delivery_events["event_timestamp"] = parse_dates(
    df_delivery_events["event_timestamp"]
)

print("Null timestamps:", df_delivery_events["event_timestamp"].isna().sum())
print(
    "Range:",
    df_delivery_events["event_timestamp"].min(),
    "to",
    df_delivery_events["event_timestamp"].max(),
)


Null timestamps: 391
Range: 2022-12-31 06:39:11 to 2026-07-01 00:00:00


In [21]:
df_delivery_events["location_city"] = clean_city(
    df_delivery_events["location_city"]
)

print(df_delivery_events["location_city"].value_counts(dropna=False))


location_city
Delhi                 1881
Kochi                  957
Kolkata                957
Pune                   956
Gurugram               954
Nagpur                 952
Surat                  951
Madurai                948
Warangal               946
Jaipur                 946
Patna                  943
Vadodara               942
Dehradun               940
Vijayawada             940
Ranchi                 940
Coimbatore             935
Ludhiana               932
Mysuru                 921
Lucknow                921
Faridabad              921
Nashik                 920
Kanpur                 917
Ahmedabad              916
Agra                   913
Bhubaneswar            913
Jodhpur                912
Guwahati               911
Chennai                910
Visakhapatnam          910
Bengaluru              907
Hyderabad              903
Rajkot                 903
Bhopal                 901
Mumbai                 894
Noida                  891
Howrah                 890
Thiruvananthap

In [22]:
df_delivery_events["remarks"] = clean_text(df_delivery_events["remarks"])

print(df_delivery_events["remarks"].value_counts(dropna=False))


remarks
<NA>                    10323
Customer unavailable     5357
On time                  5310
Address issue            5302
Traffic delay            5285
Weather delay            5251
Rescheduled              5172
Name: count, dtype: Int64


## Drivers

In [23]:
df_drivers = get_bronze("drivers")

print("Rows:", len(df_drivers))
print("Columns:", list(df_drivers.columns))
df_drivers.head()


Rows: 2030
Columns: ['driver_id', 'driver_name', 'gender', 'phone', 'license_number', 'city', 'experience_years', 'driver_rating', 'join_date']


,driver_id,driver_name,gender,phone,license_number,city,experience_years,driver_rating,join_date
0,DRV01357,Parth Sanghvi,M,6137260029,PB356107930,Ludhiana,-23,2.8,2021-09-30
1,DRV00985,Amol Subramaniam,M,9638749656,MP867496065,Bhopal,8,4.2,2025-10-29
2,DRV00860,Sai Rattan,M,7264721216,AS664868519,Guwahati,17,4.6,2023-02-06
3,DRV01985,Nakul Chand,M,9195357315,MH341005611,Nashik,12,4.5,2024-09-19
4,DRV01294,Lopa Bera,F,6676744902,PB614453094,Amritsar,11,4.2,2024-09-18


In [24]:
# Driver IDs look like DRV01357.
df_drivers["driver_id"] = clean_id(df_drivers["driver_id"], r"^DRV\d{5}$")
df_drivers = remove_duplicates(df_drivers, "driver_id")

print("Rows after removing duplicates:", len(df_drivers))
print("Rows with no driver ID:", df_drivers["driver_id"].isna().sum())


Rows after removing duplicates: 2000
Rows with no driver ID: 21


In [25]:
gender = df_drivers["gender"].astype("string").str.strip().str.lower()
gender = gender.replace({
    "m": "Male",
    "male": "Male",
    "f": "Female",
    "female": "Female",
    "other": "Others",
    "others": "Others",
})
gender.loc[~gender.isin(["Male", "Female", "Others"])] = pd.NA
df_drivers["gender"] = gender

print(df_drivers["gender"].value_counts(dropna=False))


gender
Male      1510
Female     490
Name: count, dtype: Int64


In [26]:
df_drivers["phone"] = clean_phone(df_drivers["phone"])

print("Valid phone numbers:", df_drivers["phone"].notna().sum())
print("Null phone numbers:", df_drivers["phone"].isna().sum())


Valid phone numbers: 1945
Null phone numbers: 55


In [27]:
# Licence format in this dataset: 2 letters followed by 9 digits.
df_drivers["license_number"] = clean_id(
    df_drivers["license_number"], r"^[A-Z]{2}\d{9}$"
)

print("Valid licences:", df_drivers["license_number"].notna().sum())
print("Null licences:", df_drivers["license_number"].isna().sum())


Valid licences: 1935
Null licences: 65


In [28]:
df_drivers["city"] = clean_city(df_drivers["city"])

print(df_drivers["city"].value_counts(dropna=False))


city
Delhi                 113
Hubballi               58
Bhopal                 57
Chandigarh             56
Faridabad              53
Coimbatore             52
Guwahati               51
Varanasi               51
Patna                  51
Udaipur                51
Kochi                  50
Surat                  50
Ranchi                 50
Nagpur                 50
Howrah                 50
Hyderabad              48
Jaipur                 47
Gurugram               46
Kolkata                46
Visakhapatnam          45
Bengaluru              45
Ludhiana               44
Mumbai                 44
Siliguri               44
Noida                  43
Bhubaneswar            42
Chennai                42
Jodhpur                41
Madurai                41
Nashik                 40
Indore                 40
Ahmedabad              39
Vadodara               38
Thiruvananthapuram     38
Rajkot                 37
Vijayawada             36
Dehradun               36
Warangal               35
Mysuru 

In [29]:
df_drivers["experience_years"] = pd.to_numeric(
    df_drivers["experience_years"], errors="coerce"
)
df_drivers.loc[df_drivers["experience_years"] < 0, "experience_years"] = pd.NA

print("Null experience:", df_drivers["experience_years"].isna().sum())
print(
    "Range:",
    df_drivers["experience_years"].min(),
    "to",
    df_drivers["experience_years"].max(),
)


Null experience: 61
Range: 0.0 to 25.0


In [30]:
df_drivers["driver_rating"] = pd.to_numeric(
    df_drivers["driver_rating"], errors="coerce"
)
df_drivers.loc[
    ~df_drivers["driver_rating"].between(0, 5), "driver_rating"
] = pd.NA

print("Null ratings:", df_drivers["driver_rating"].isna().sum())
print(
    "Range:",
    df_drivers["driver_rating"].min(),
    "to",
    df_drivers["driver_rating"].max(),
)


Null ratings: 60
Range: 2.5 to 5.0


In [31]:
df_drivers["join_date"] = parse_dates(df_drivers["join_date"])

print("Null join dates:", df_drivers["join_date"].isna().sum())
print(
    "Date range:",
    df_drivers["join_date"].min(),
    "to",
    df_drivers["join_date"].max(),
)


Null join dates: 22
Date range: 2018-01-01 00:00:00 to 2027-11-22 00:00:00


## Routes

In [32]:
df_routes = get_bronze("routes")

print("Rows:", len(df_routes))
print("Columns:", list(df_routes.columns))
df_routes.head()


Rows: 1200
Columns: ['route_id', 'origin_city', 'origin_state', 'destination_city', 'destination_state', 'distance_km', 'avg_duration_hr']


,route_id,origin_city,origin_state,destination_city,destination_state,distance_km,avg_duration_hr
0,RT00001,Ranchi,Jharkhand,Nagpur,Maharashtra,1174.4,29.5
1,RT00002,Guwahati,Assam,Warangal,Telangana,-531.5,11.3
2,RT00003,Indore,Madhya Pradesh,Kanpur,Uttar Pradesh,1287.9,31.2
3,RT00004,Nashik,Maharashtra,Chandigarh,Chandigarh,2174.9,50.6
4,RT00005,Udaipur,Rajasthan,FARIDABAD,Haryana,1615.0,33.6 hr


In [33]:
# Route IDs look like RT00001.
df_routes["route_id"] = clean_id(df_routes["route_id"], r"^RT\d{5}$")
df_routes = remove_duplicates(df_routes, "route_id")

print("Rows:", len(df_routes))


Rows: 1200


In [34]:
df_routes["origin_city"] = clean_city(df_routes["origin_city"])
df_routes["origin_state"] = clean_state(df_routes["origin_state"])

print(df_routes["origin_city"].value_counts(dropna=False))
print(df_routes["origin_state"].value_counts(dropna=False))


origin_city
Delhi                 47
Nagpur                38
Jodhpur               38
Vijayawada            37
Surat                 33
Rajkot                33
Ahmedabad             32
Patna                 32
Hyderabad             32
Ranchi                31
Vadodara              31
Bhopal                30
Indore                29
Bhubaneswar           29
Visakhapatnam         29
Dehradun              29
Udaipur               28
Kochi                 28
Noida                 28
Ludhiana              28
Varanasi              27
Mysuru                27
Kanpur                27
Bengaluru             26
Agra                  26
Howrah                26
Siliguri              26
Pune                  26
Jaipur                25
Warangal              25
Mumbai                24
Thiruvananthapuram    23
Hubballi              23
Nashik                22
Gurugram              22
Lucknow               22
Amritsar              21
Madurai               21
Faridabad             21
Chennai      

In [35]:
df_routes["destination_city"] = clean_city(df_routes["destination_city"])
df_routes["destination_state"] = clean_state(df_routes["destination_state"])

print(df_routes["destination_city"].value_counts(dropna=False))
print(df_routes["destination_state"].value_counts(dropna=False))


destination_city
Delhi                 51
Indore                39
Kochi                 36
Bhubaneswar           33
Udaipur               33
Lucknow               33
Hyderabad             31
Bhopal                31
Warangal              30
Chandigarh            30
Vijayawada            30
Jaipur                30
Varanasi              30
Siliguri              30
Kanpur                29
Pune                  29
Coimbatore            28
Ranchi                28
Kolkata               28
Patna                 28
Chennai               27
Noida                 27
Nagpur                26
Nashik                26
Agra                  26
Surat                 26
Howrah                26
Ludhiana              25
Hubballi              25
Thiruvananthapuram    25
Faridabad             24
Gurugram              24
Amritsar              24
Rajkot                24
Ahmedabad             24
Vadodara              24
Guwahati              23
Mysuru                22
Jodhpur               21
Mumbai  

In [36]:
df_routes["distance_km"] = pd.to_numeric(
    df_routes["distance_km"], errors="coerce"
)
df_routes.loc[df_routes["distance_km"] <= 0, "distance_km"] = pd.NA

print("Null distances:", df_routes["distance_km"].isna().sum())
print("Range:", df_routes["distance_km"].min(), "to", df_routes["distance_km"].max())


Null distances: 69
Range: 20.3 to 78257.6


In [37]:
df_routes["avg_duration_hr"] = to_number(
    df_routes["avg_duration_hr"], unit="hr"
)
df_routes.loc[
    df_routes["avg_duration_hr"] <= 0, "avg_duration_hr"
] = pd.NA

print("Null durations:", df_routes["avg_duration_hr"].isna().sum())
print(
    "Range:",
    df_routes["avg_duration_hr"].min(),
    "to",
    df_routes["avg_duration_hr"].max(),
)


Null durations: 7
Range: 0.4 to 70.9


## Vehicles

In [38]:
df_vehicles = get_bronze("vehicles")

print("Rows:", len(df_vehicles))
print("Columns:", list(df_vehicles.columns))
df_vehicles.head()


Rows: 1515
Columns: ['vehicle_id', 'vehicle_number', 'vehicle_type', 'make', 'year', 'capacity_kg', 'status']


,vehicle_id,vehicle_number,vehicle_type,make,year,capacity_kg,status
0,VEH00052,UP45YA5539,mini truck,Mahindra,2012,1496,Active
1,VEH00169,tn84qw0499,Mini Truck,Force Motors,2014,837,Active
2,VEH01469,WB10IT1846,mini truck,Tata,2025,839,Retired
3,VEH00927,WB84QX6779,Tempo,Piaggio,2019,2970,Active
4,VEH00423,OD17WS3915,Pickup Van,Piaggio,2014,999,NaN


In [39]:
# Vehicle IDs look like VEH00052.
df_vehicles["vehicle_id"] = clean_id(df_vehicles["vehicle_id"], r"^VEH\d{5}$")
df_vehicles = remove_duplicates(df_vehicles, "vehicle_id")

print("Rows after removing duplicates:", len(df_vehicles))


Rows after removing duplicates: 1500


In [40]:
# Indian registration: 2 letters, 2 digits, 1-3 letters, 4 digits.
vehicle_number = df_vehicles["vehicle_number"].astype("string").str.strip().str.upper()
valid = vehicle_number.str.match(r"^[A-Z]{2}\d{2}[A-Z]{1,3}\d{4}$", na=False)
vehicle_number.loc[~valid] = pd.NA
df_vehicles["vehicle_number"] = vehicle_number

print("Null vehicle numbers:", df_vehicles["vehicle_number"].isna().sum())
print(
    "Duplicate numbers:",
    df_vehicles["vehicle_number"].dropna().duplicated().sum(),
)


Null vehicle numbers: 0
Duplicate numbers: 0


In [41]:
vehicle_type = df_vehicles["vehicle_type"].astype("string").str.strip().str.lower()
vehicle_type = vehicle_type.replace({
    "mini-truck": "Mini Truck",
    "mini truck": "Mini Truck",
    "pickup-van": "Pickup Van",
    "pickup van": "Pickup Van",
    "pickup": "Pickup Van",
    "2 wheeler": "Two-Wheeler",
    "two-wheeler": "Two-Wheeler",
    "3-wheeler": "Three-Wheeler",
    "three wheeler": "Three-Wheeler",
    "three-wheeler": "Three-Wheeler",
    "container-truck": "Container Truck",
    "container truck": "Container Truck",
    "container": "Container Truck",
    "tempo": "Tempo",
    "trailer": "Trailer",
})

valid_vehicle_types = [
    "Mini Truck",
    "Pickup Van",
    "Two-Wheeler",
    "Three-Wheeler",
    "Container Truck",
    "Tempo",
    "Trailer",
]
vehicle_type.loc[~vehicle_type.isin(valid_vehicle_types)] = pd.NA
df_vehicles["vehicle_type"] = vehicle_type

print(df_vehicles["vehicle_type"].value_counts(dropna=False))


vehicle_type
Container Truck    227
Three-Wheeler      221
Tempo              218
Trailer            215
Mini Truck         208
Two-Wheeler        208
Pickup Van         198
<NA>                 5
Name: count, dtype: Int64


In [42]:
df_vehicles["make"] = clean_text(df_vehicles["make"]).str.title()

print(df_vehicles["make"].value_counts(dropna=False))


make
Eicher           236
Bajaj            220
Piaggio          214
Ashok Leyland    212
Force Motors     209
Tata             205
Mahindra         204
Name: count, dtype: Int64


In [43]:
df_vehicles["year"] = pd.to_numeric(df_vehicles["year"], errors="coerce")

current_year = pd.Timestamp.today().year
df_vehicles.loc[
    (df_vehicles["year"] < 1998) | (df_vehicles["year"] > current_year),
    "year",
] = pd.NA
df_vehicles["year"] = df_vehicles["year"].astype("Int64")

print("Null years:", df_vehicles["year"].isna().sum())
print("Range:", df_vehicles["year"].min(), "to", df_vehicles["year"].max())


Null years: 30
Range: 1998 to 2025


In [44]:
df_vehicles["capacity_kg"] = to_number(df_vehicles["capacity_kg"], unit="kg")
df_vehicles.loc[df_vehicles["capacity_kg"] <= 0, "capacity_kg"] = pd.NA

print("Null capacities:", df_vehicles["capacity_kg"].isna().sum())
print(
    "Range:",
    df_vehicles["capacity_kg"].min(),
    "to",
    df_vehicles["capacity_kg"].max(),
)


Null capacities: 51
Range: 5.0 to 29771.0


In [45]:
status = df_vehicles["status"].astype("string").str.strip().str.lower()
status = status.replace({
    "active": "Active",
    "retired": "Retired",
    "under maintenance": "Under Maintenance",
})
status.loc[~status.isin(["Active", "Retired", "Under Maintenance"])] = pd.NA
df_vehicles["status"] = status

print(df_vehicles["status"].value_counts(dropna=False))


status
Active               889
Under Maintenance    305
Retired              296
<NA>                  10
Name: count, dtype: Int64


## Shipments

In [46]:
df_shipments = get_bronze("shipments")

print("Rows:", len(df_shipments))
print("Columns:", list(df_shipments.columns))
df_shipments.head()


Rows: 91080
Columns: ['shipment_id', 'customer_id', 'warehouse_id', 'driver_id', 'vehicle_id', 'route_id', 'shipment_date', 'expected_delivery_date', 'actual_delivery_date', 'package_type', 'weight_kg', 'distance_km', 'shipping_cost_inr', 'payment_method', 'shipment_status', 'delivery_rating']


,shipment_id,customer_id,warehouse_id,driver_id,vehicle_id,route_id,shipment_date,expected_delivery_date,actual_delivery_date,package_type,weight_kg,distance_km,shipping_cost_inr,payment_method,shipment_status,delivery_rating
0,SHP0059521,CUST012290,WH0038,DRV01459,VEH00332,RT00657,2023-04-19,2023-04-29,2023-04-30,Pharma,319.51,431.9,4141.65,UPI,Delivered,4
1,SHP0089749,CUST003285,WH0196,DRV00532,VEH01056,RT00043,2025-06-15,2025-06-24,NaN,FMCG,406.21,15561.5,4979.85,Net Banking,Pending,5
2,SHP0012812,CUST008545,WH0062,DRV01839,VEH01495,RT00428,2025-10-08,2025-10-10,2025-10-11,Documents,168.52,898.5,2855.87,Net Banking,Delivered,3
3,SHP0032190,CUST011305,WH0016,DRV00655,VEH01064,RT01064,2025-10-05,2025-10-11,NaN,Industrial Equipment,478.28,1870.6,8364.79,CREDIT CARD,In Transit,2
4,SHP0052788,CUST000947,WH0192,DRV00842,VEH01384,RT00454,2025-02-20,2025-02-23,NaN,Apparel,247.7,693.2,5325.37,Net Banking,In Transit,5


In [47]:
# Check every ID column in one place.
# IDs look like SHP0059521, CUST004292, WH0096, DRV01357, VEH00052, RT00001.
df_shipments["shipment_id"] = clean_id(
    df_shipments["shipment_id"], r"^SHP\d{7}$"
)
df_shipments["customer_id"] = clean_id(
    df_shipments["customer_id"], r"^CUST\d{6}$"
)
df_shipments["warehouse_id"] = clean_id(
    df_shipments["warehouse_id"], r"^WH\d{4}$"
)
df_shipments["driver_id"] = clean_id(
    df_shipments["driver_id"], r"^DRV\d{5}$"
)
df_shipments["vehicle_id"] = clean_id(
    df_shipments["vehicle_id"], r"^VEH\d{5}$"
)
df_shipments["route_id"] = clean_id(
    df_shipments["route_id"], r"^RT\d{5}$"
)

for column in [
    "shipment_id",
    "customer_id",
    "warehouse_id",
    "driver_id",
    "vehicle_id",
    "route_id",
]:
    print(column, "nulls:", df_shipments[column].isna().sum())


shipment_id nulls: 0
customer_id nulls: 1059
warehouse_id nulls: 0
driver_id nulls: 942
vehicle_id nulls: 0
route_id nulls: 0


In [48]:
df_shipments["shipment_date"] = parse_dates(df_shipments["shipment_date"])

# A shipment date cannot be in the future.
df_shipments.loc[
    df_shipments["shipment_date"] > today, "shipment_date"
] = pd.NaT

print("Null shipment dates:", df_shipments["shipment_date"].isna().sum())


Null shipment dates: 933


In [49]:
df_shipments["expected_delivery_date"] = parse_dates(
    df_shipments["expected_delivery_date"]
)

too_early = (
    df_shipments["expected_delivery_date"]
    < df_shipments["shipment_date"]
)
too_late = (
    df_shipments["expected_delivery_date"]
    > df_shipments["shipment_date"] + pd.Timedelta(days=365)
)
df_shipments.loc[too_early | too_late, "expected_delivery_date"] = pd.NaT

print(
    "Null expected dates:",
    df_shipments["expected_delivery_date"].isna().sum(),
)


Null expected dates: 2070


In [50]:
df_shipments["actual_delivery_date"] = parse_dates(
    df_shipments["actual_delivery_date"]
)

# An actual delivery date cannot be in the future.
df_shipments.loc[
    df_shipments["actual_delivery_date"] > today, "actual_delivery_date"
] = pd.NaT

too_early = (
    df_shipments["actual_delivery_date"] < df_shipments["shipment_date"]
)
too_late = (
    df_shipments["actual_delivery_date"]
    > df_shipments["shipment_date"] + pd.Timedelta(days=365)
)
df_shipments.loc[too_early | too_late, "actual_delivery_date"] = pd.NaT

print(
    "Null actual dates:",
    df_shipments["actual_delivery_date"].isna().sum(),
)


Null actual dates: 47025


In [51]:
package_type = df_shipments["package_type"].astype("string").str.strip().str.lower()
package_type = package_type.replace({
    "fmcg": "FMCG",
    "documents": "Documents",
    "auto parts": "Auto Parts",
    "industrial equipment": "Industrial Equipment",
    "pharma": "Pharma",
    "furniture": "Furniture",
    "apparel": "Apparel",
    "groceries": "Groceries",
    "electronics": "Electronics",
})

valid_packages = [
    "FMCG",
    "Documents",
    "Auto Parts",
    "Industrial Equipment",
    "Pharma",
    "Furniture",
    "Apparel",
    "Groceries",
    "Electronics",
]
package_type.loc[~package_type.isin(valid_packages)] = pd.NA
df_shipments["package_type"] = package_type

print(df_shipments["package_type"].value_counts(dropna=False))


package_type
FMCG                    10119
Documents               10101
Auto Parts              10089
Industrial Equipment    10030
Pharma                  10021
Furniture                9995
Apparel                  9978
Groceries                9917
Electronics              9862
<NA>                      968
Name: count, dtype: Int64


In [52]:
df_shipments["weight_kg"] = to_number(df_shipments["weight_kg"], unit="kg")
df_shipments.loc[df_shipments["weight_kg"] <= 0, "weight_kg"] = pd.NA

print("Null weights:", df_shipments["weight_kg"].isna().sum())
print(
    "Range:",
    df_shipments["weight_kg"].min(),
    "to",
    df_shipments["weight_kg"].max(),
)


Null weights: 3749
Range: 0.5 to 19991.66


In [53]:
df_shipments["distance_km"] = pd.to_numeric(
    df_shipments["distance_km"], errors="coerce"
)
df_shipments.loc[df_shipments["distance_km"] <= 0, "distance_km"] = pd.NA

print("Null distances:", df_shipments["distance_km"].isna().sum())
print(
    "Range:",
    df_shipments["distance_km"].min(),
    "to",
    df_shipments["distance_km"].max(),
)


Null distances: 1793
Range: 5.0 to 61561.3


In [54]:
df_shipments["shipping_cost_inr"] = to_number(
    df_shipments["shipping_cost_inr"], keep_currency=True
)
df_shipments.loc[
    df_shipments["shipping_cost_inr"] <= 0, "shipping_cost_inr"
] = pd.NA

print("Null costs:", df_shipments["shipping_cost_inr"].isna().sum())
print(
    "Range:",
    df_shipments["shipping_cost_inr"].min(),
    "to",
    df_shipments["shipping_cost_inr"].max(),
)


Null costs: 2775
Range: 17.78 to 721130.94


In [55]:
payment_method = df_shipments["payment_method"].astype("string").str.strip().str.lower()
payment_method = payment_method.replace({
    "upi": "UPI",
    "u.p.i": "UPI",
    "net banking": "Net Banking",
    "netbanking": "Net Banking",
    "net-banking": "Net Banking",
    "credit card": "Credit Card",
    "credit-card": "Credit Card",
    "creditcard": "Credit Card",
    "debit card": "Debit Card",
    "debitcard": "Debit Card",
    "cod": "Cash on Delivery",
    "c.o.d": "Cash on Delivery",
    "cash on delivery": "Cash on Delivery",
    "wallet": "Wallet",
    "e-wallet": "Wallet",
    "prepaid": "Prepaid",
    "pre-paid": "Prepaid",
    "pre paid": "Prepaid",
})

valid_payments = [
    "UPI",
    "Net Banking",
    "Credit Card",
    "Debit Card",
    "Cash on Delivery",
    "Wallet",
    "Prepaid",
]
payment_method.loc[~payment_method.isin(valid_payments)] = pd.NA
df_shipments["payment_method"] = payment_method

print(df_shipments["payment_method"].value_counts(dropna=False))


payment_method
UPI                 12996
Prepaid             12996
Debit Card          12988
Net Banking         12934
Cash on Delivery    12933
Wallet              12910
Credit Card         12896
<NA>                  427
Name: count, dtype: Int64


In [56]:
shipment_status = (
    df_shipments["shipment_status"].astype("string").str.strip().str.lower()
)
shipment_status = shipment_status.replace({
    "delivered": "Delivered",
    "deliverd": "Delivered",
    "deliv erd": "Delivered",
    "return": "Returned",
    "retured": "Returned",
    "returned": "Returned",
    "in transit": "In Transit",
    "in-transit": "In Transit",
    "in_transit": "In Transit",
    "intransit": "In Transit",
    "cancled": "Cancelled",
    "canceled": "Cancelled",
    "cancelled!": "Cancelled",
    "cancelled": "Cancelled",
    "delayd": "Delayed",
    "delayed": "Delayed",
    "pendng": "Pending",
    "pending": "Pending",
})

valid_statuses = [
    "Delivered",
    "Returned",
    "In Transit",
    "Cancelled",
    "Delayed",
    "Pending",
]
shipment_status.loc[~shipment_status.isin(valid_statuses)] = pd.NA
df_shipments["shipment_status"] = shipment_status

print(df_shipments["shipment_status"].value_counts(dropna=False))


shipment_status
In Transit    15179
Returned      15143
Cancelled     15138
Delayed       15090
Pending       15052
Delivered     14889
<NA>            589
Name: count, dtype: Int64


In [57]:
df_shipments["delivery_rating"] = pd.to_numeric(
    df_shipments["delivery_rating"], errors="coerce"
)
df_shipments.loc[
    ~df_shipments["delivery_rating"].between(1, 5), "delivery_rating"
] = pd.NA
df_shipments["delivery_rating"] = df_shipments["delivery_rating"].astype("Int64")

print(df_shipments["delivery_rating"].value_counts(dropna=False))


delivery_rating
5       25548
4       25534
3       12974
2       12897
1       12716
<NA>     1411
Name: count, dtype: Int64


In [58]:
df_shipments = remove_duplicates(df_shipments, "shipment_id")

print("Rows after removing duplicates:", len(df_shipments))
print(
    "Duplicate shipment IDs:",
    df_shipments["shipment_id"].dropna().duplicated().sum(),
)


Rows after removing duplicates: 90000
Duplicate shipment IDs: 0


## Warehouses

In [59]:
df_warehouses = get_bronze("warehouses")

print("Rows:", len(df_warehouses))
print("Columns:", list(df_warehouses.columns))
df_warehouses.head()


Rows: 202
Columns: ['warehouse_id', 'warehouse_name', 'city', 'state', 'pincode', 'capacity_units', 'manager_name', 'operational_since']


,warehouse_id,warehouse_name,city,state,pincode,capacity_units,manager_name,operational_since
0,WH0096,Bhopal Logistics Hub 96,Bhopal,Madhya Pradesh,714650,7301,Avi Kala,2018-12-20
1,WH0016,Mysuru Logistics Hub 16,Mysuru,Karnataka,666174,10043,Jagrati Goswami,10-20-2024
2,WH0031,Chandigarh Logistics Hub 31,Chandigarh,Chandigarh,307815,18123,Atharv Setty,2021-08-17
3,WH0160,Warangal Logistics Hub 160,Warangal,Telangana,298486,7453,Anita Bhalla,2022-09-06
4,WH0187,Chennai Logistics Hub 187,Chennai,Tamil Nadu,272935,19959,Yahvi Kari,2020-11-17


In [60]:
# Warehouse IDs look like WH0096.
df_warehouses["warehouse_id"] = clean_id(
    df_warehouses["warehouse_id"], r"^WH\d{4}$"
)
df_warehouses = remove_duplicates(df_warehouses, "warehouse_id")

print("Rows after removing duplicates:", len(df_warehouses))


Rows after removing duplicates: 200


In [61]:
warehouse_name = clean_text(df_warehouses["warehouse_name"])
# Turn many spaces into one space.
warehouse_name = warehouse_name.str.replace(r"\s+", " ", regex=True)
df_warehouses["warehouse_name"] = warehouse_name.str.title()

print(df_warehouses["warehouse_name"].value_counts(dropna=False))


warehouse_name
Bhopal Logistics Hub 96                1
Mysuru Logistics Hub 16                1
Chandigarh Logistics Hub 31            1
Warangal Logistics Hub 160             1
Chennai Logistics Hub 187              1
                                      ..
Madurai Logistics Hub 107              1
Thiruvananthapuram Logistics Hub 15    1
Pune Logistics Hub 93                  1
Mumbai Logistics Hub 180               1
Nashik Logistics Hub 103               1
Name: count, Length: 200, dtype: Int64


In [62]:
df_warehouses["city"] = clean_city(df_warehouses["city"])
df_warehouses["state"] = clean_state(df_warehouses["state"])

print(df_warehouses["city"].value_counts(dropna=False))
print(df_warehouses["state"].value_counts(dropna=False))


city
Hubballi              11
Mumbai                 8
Madurai                8
Udaipur                8
Hyderabad              8
Bhopal                 7
Chandigarh             7
Chennai                7
Nashik                 6
Vadodara               6
Kolkata                6
Guwahati               5
Gurugram               5
Kochi                  5
Indore                 5
Rajkot                 5
Faridabad              5
Agra                   5
Siliguri               5
Patna                  4
Vijayawada             4
Jodhpur                4
Coimbatore             4
Surat                  4
Lucknow                4
Noida                  4
Dehradun               4
Pune                   4
Ludhiana               4
Kanpur                 4
Thiruvananthapuram     3
Bengaluru              3
Delhi                  3
Jaipur                 3
Ranchi                 3
Amritsar               3
Nagpur                 3
Mysuru                 2
Warangal               2
Ahmedabad           

In [63]:
# Keep only digits, then check for a valid 6-digit pincode.
pincode = df_warehouses["pincode"].astype("string").str.replace(
    r"[^0-9]", "", regex=True
)
valid = pincode.str.match(r"^[1-9]\d{5}$", na=False)
pincode.loc[~valid] = pd.NA
df_warehouses["pincode"] = pincode

print("Null pincodes:", df_warehouses["pincode"].isna().sum())


Null pincodes: 7


In [64]:
df_warehouses["capacity_units"] = pd.to_numeric(
    df_warehouses["capacity_units"], errors="coerce"
)
df_warehouses.loc[
    df_warehouses["capacity_units"] <= 0, "capacity_units"
] = pd.NA
df_warehouses["capacity_units"] = df_warehouses["capacity_units"].astype("Int64")

print("Null capacities:", df_warehouses["capacity_units"].isna().sum())
print(
    "Range:",
    df_warehouses["capacity_units"].min(),
    "to",
    df_warehouses["capacity_units"].max(),
)


Null capacities: 9
Range: 1358 to 19959


In [65]:
manager_name = clean_text(df_warehouses["manager_name"])
# Turn many spaces into one space.
manager_name = manager_name.str.replace(r"\s+", " ", regex=True)
df_warehouses["manager_name"] = manager_name.str.title()

print(df_warehouses["manager_name"].value_counts(dropna=False))


manager_name
Avi Kala            1
Jagrati Goswami     1
Atharv Setty        1
Anita Bhalla        1
Yahvi Kari          1
                   ..
Widisha Zacharia    1
Agastya Sarkar      1
Ekaja De            1
Ucchal Mangal       1
Bhavika Cheema      1
Name: count, Length: 200, dtype: Int64


In [66]:
df_warehouses["operational_since"] = parse_dates(
    df_warehouses["operational_since"]
)

print(
    "Null dates:",
    df_warehouses["operational_since"].isna().sum(),
)
print(
    "Date range:",
    df_warehouses["operational_since"].min(),
    "to",
    df_warehouses["operational_since"].max(),
)


Null dates: 2
Date range: 2015-01-01 00:00:00 to 2024-10-20 00:00:00


## Link checks and load

Before loading, remove links that point to IDs that do not exist in the
other tables.


In [67]:
# Shipments: remove IDs that are not in the matching dimension table.
for column, valid_values in [
    ("customer_id", df_customers["customer_id"]),
    ("driver_id", df_drivers["driver_id"]),
    ("vehicle_id", df_vehicles["vehicle_id"]),
    ("route_id", df_routes["route_id"]),
    ("warehouse_id", df_warehouses["warehouse_id"]),
]:
    missing = df_shipments[column].notna() & ~df_shipments[column].isin(
        valid_values
    )
    df_shipments.loc[missing, column] = pd.NA
    print(column, "orphans removed:", missing.sum())


customer_id orphans removed: 0
driver_id orphans removed: 1056
vehicle_id orphans removed: 698
route_id orphans removed: 697
warehouse_id orphans removed: 729


In [68]:
# Delivery events: remove shipment IDs that are not in shipments.
missing = (
    df_delivery_events["shipment_id"].notna()
    & ~df_delivery_events["shipment_id"].isin(df_shipments["shipment_id"])
)
df_delivery_events.loc[missing, "shipment_id"] = pd.NA
print("delivery_events.shipment_id orphans removed:", missing.sum())


delivery_events.shipment_id orphans removed: 621


In [69]:
load_to_silver(df_customers, "customers")
load_to_silver(df_drivers, "drivers")
load_to_silver(df_routes, "routes")
load_to_silver(df_vehicles, "vehicles")
load_to_silver(df_warehouses, "warehouses")
load_to_silver(df_shipments, "shipments")
load_to_silver(df_delivery_events, "delivery_events")


Loaded silver.customers
Loaded silver.drivers
Loaded silver.routes
Loaded silver.vehicles
Loaded silver.warehouses
Loaded silver.shipments
Loaded silver.delivery_events


In [71]:
tables = {
    "customers": "customer_id",
    "delivery_events": "event_id",
    "drivers": "driver_id",
    "routes": "route_id",
    "shipments": "shipment_id",
    "vehicles": "vehicle_id",
    "warehouses": "warehouse_id",
}

print("SILVER VALIDATION")
print("-" * 60)

for table, key in tables.items():
    df = pd.read_sql(f"select * from silver.{table}", engine)
    print(
        f"{table}: rows={len(df)}, "
        f"null IDs={df[key].isna().sum()}, "
        f"duplicate IDs={df[key].dropna().duplicated().sum()}"
    )


SILVER VALIDATION
------------------------------------------------------------
customers: rows=15004, null IDs=225, duplicate IDs=0
delivery_events: rows=42000, null IDs=0, duplicate IDs=0
drivers: rows=2000, null IDs=21, duplicate IDs=0
routes: rows=1200, null IDs=0, duplicate IDs=0
shipments: rows=90000, null IDs=0, duplicate IDs=0
vehicles: rows=1500, null IDs=0, duplicate IDs=0
warehouses: rows=200, null IDs=0, duplicate IDs=0
